# Porovnání plánů splácení studentských půjček pomocí PROC LOAN

## Shrnutí

Finanční oddělení podpory studentů posuzuje, jak by absolventská kohorta měla splácet reprezentativní zůstatek 27 500 USD porovnáním čtyř splátkových struktur — federálního fixního standardního plánu, soukromého refinancování se zřizovacím poplatkem, úvěru s pohyblivou sazbou (ARM) s omezením a zaměstnavatelsky dotovaného dočasného snížení sazby (buydown) — pomocí `PROC LOAN`. Během 120měsíční doby splácení se čtyři nabídky podle uvedené počáteční sazby jasně liší v měsíční splátce a celkovém úroku: federální standardní plán stojí na úroku nejvíc (**10 021,22 USD** při 6,53 %, splátka **312,68 USD**), soukromé refinancování snižuje sazbu, ale přidává poplatek 412,50 USD (**9 120,20 USD** při 5,99 %, splátka **305,17 USD**), a nižší nabízený ARM (**7 099,76 USD** při 4,75 %, splátka **288,33 USD**) a zaměstnavatelský buydown (**6 700,67 USD** při 4,50 %, splátka **285,01 USD**) mají nejnižší úrokové náklady. Blok `COMPARE` pak uvádí kumulativní úrok, jistinu a nesplacený zůstatek každého plánu ve 36., 60. a 120. měsíci, což ukazuje, jak daleko je každý úvěr amortizován v bodech, kdy dlužník nejspíš refinancuje nebo splatí zbytek.

## Zdroje dat

| Datová sada | Řádky | Popis | Klíčové proměnné |
|---------|------|-------------|---------------|
| `borrowers` | 40 | Syntetické profily úvěrů absolventské kohorty vygenerované přímo v kódu pomocí `call streaminit(20260531)` a `rand('uniform')`. Slouží k odvození realistických podmínek úvěru pro porovnání. | `student_id` (1001-1040), `balance` (jistina při absolvování; tento vzorek se pohybuje od 11 800 USD do 47 750 USD, průměr 30 771 USD), `apr` (roční nominální sazba 4,10 %-9,10 %, průměr 6,50 %), `term` (120 nebo 180 měsíců, průměr 144), `origination_pct` (poplatek 1,0 %-2,0 %, průměr 1,50 %) |

Reprezentativní dlužník analyzovaný pomocí `PROC LOAN` (částka 27 500 USD, doba splácení 120 měsíců, začátek červenec 2026) se nachází v dolní střední části rozložení zůstatků a sazeb této kohorty; nepoužívají se žádná externí ani síťová data. Kohorta slouží k odvození realistických podmínek úvěru — samotné porovnání se provádí na jediném reprezentativním úvěru.

# Porovnání plánů splácení studentských půjček pomocí PROC LOAN

Když studenti absolvují, musí jim finanční oddělení podpory pomoci vybrat mezi konkurenčními nabídkami splácení. `PROC LOAN` (SAS/ETS) je určen přesně k tomuto účelu: amortizuje úvěry s pevnou sazbou, s pohyblivou sazbou (ARM) a s dočasně sníženou sazbou (buydown) a poté je porovná vedle sebe podle splátky, celkového úroku a průběhu amortizace.

V tomto sešitu:

1. Vygenerujeme syntetickou absolventskou kohortu pro stanovení realistických podmínek úvěru.
2. Shrneme kohortu pomocí `PROC MEANS`.
3. Sestavíme čtyři splátkové plány pro reprezentativní zůstatek 27 500 USD a porovnáme je pomocí `PROC LOAN` (FIXED / ARM / BUYDOWN + COMPARE).
4. Znovu spustíme doporučený plán s pevnou sazbou samostatně, abychom potvrdili jeho samostatnou ekonomiku.

Vše běží lokálně na interních syntetických datech — bez externích souborů nebo síťového přístupu.

## Krok 1 — Generování syntetické absolventské kohorty

Simulujeme 40 dlužníků. Každý získá jistinu při absolvování, roční sazbu APR volně vázanou na úvěrovou kvalitu, standardní dobu splácení (10 nebo 15 let) a procento zřizovacího poplatku. Počáteční hodnota generátoru zajišťuje reprodukovatelnost dat.

In [1]:
data borrowers;
   CALL streaminit(20260531);
   DÉLKA plan $ 28;
   OPAKUJ student_id = 1001 TO 1040;
      /* Jistina při absolvování: 9 500 USD - 48 000 USD */
      balance = round(9500 + 38500*rand('uniform'), 50);
      /* Roční nominální APR podle úvěrové kategorie: 4,0 % - 9,5 % */
      apr = round(4.0 + 5.5*rand('uniform'), 0.05);
      /* Standardní doba splácení: 120 nebo 180 měsíců */
      KDYŽ rand('uniform') < 0.6 PAK term = 120;
      JINAK term = 180;
      /* Zřizovací poplatek jako procento jistiny: 1,0 % - 2,0 % */
      origination_pct = round(1.0 + rand('uniform'), 0.10);
      VÝSTUP;
   KONEC;
SPUSTIT;


NOTE: DATA borrowers


NOTE: Wrote borrowers (40 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## Krok 2 — Profilace kohorty

Před modelováním jednotlivých úvěrů se podíváme na rozložení zůstatků, sazeb a dob splácení. To ukazuje, jak vypadá *reprezentativní* dlužník — základ pro následné přímé porovnání.

In [2]:
PROCEDURA PRŮMĚRY data=borrowers n mean MIN MAX maxdec=2;
   ŠTÍTEK balance="Zůstatek (USD)" apr="Roční sazba APR (%)" term="Doba splácení (měsíce)"
         origination_pct="Zřizovací poplatek (%)";
   PROMĚNNÁ balance apr term origination_pct;
SPUSTIT;

                                                  The MEANS Procedure

 Variable         Label                             N           Mean     Minimum     Maximum
 -------------------------------------------------------------------------------------------
 balance          Zůstatek (USD)                   40       30771.25    11800.00    47750.00
 apr              Roční sazba APR (%)              40           6.50        4.10        9.10
 term             Doba splácení (měsíce)           40         144.00      120.00      180.00
 origination_pct  Zřizovací poplatek (%)           40           1.50        1.00        2.00
 -------------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Krok 3 — Porovnání čtyř splátkových plánů pro reprezentativního dlužníka

Pro reprezentativní zůstatek 27 500 USD během 120měsíční (10leté) doby splácení začínající v červenci 2026 porovnáme čtyři reálné nabídky:

- **Federální přímý nedotovaný úvěr (standardní)** — jednoduchý úvěr s pevnou sazbou 6,53 %.
- **Soukromé refinancování (s poplatkem)** — nižší pevná sazba 5,99 %, ale se zřizovacím nákladem 412,50 USD (`INIT=`).
- **Soukromý úvěr s proměnlivou sazbou (ARM)** — úvěr s pohyblivou sazbou 4,75 % s omezením 1 % na úpravu / 5 % za dobu trvání (`CAPS=`), maximální sazbou 9,75 % (`MAXRATE=`), ročním intervalem úpravy (`ADJUSTFREQ=`) a klíčovým slovem `WORSTCASE`.
- **Zaměstnavatelský buydown 2-1** — dotovaný start na 4,50 %, který podle uvedeného harmonogramu stoupá pomocí `BDRATES=` na 5,50 % ve 2. roce a poté na 6,50 %.

Příkaz `COMPARE` požaduje meziúvěrové srovnání ve 36., 60. a 120. měsíci, s daňovou sazbou 22 % (`TAXRATE=`) a minimální atraktivní mírou návratnosti 4 % (`MARR=`); `OUTSUM=` a `OUTCOMP=` zachycují souhrn za jednotlivé úvěry a řádky porovnání. Následující výpis uvádí pro každý plán a každý kontrolní bod **kumulativně zaplacený úrok, kumulativně splacenou jistinu a nesplacený zůstatek** — přehledný pohled na průběh amortizace napříč kandidáty.

> **Poznámka k úpravám sazby.** Jennerova `PROC LOAN` amortizuje každý plán podle jeho uvedené **počáteční** sazby, takže nastavení `CAPS`/`MAXRATE`/`WORSTCASE` u ARM a kroky `BDRATES` u buydownu se ve výpisu zobrazí jako podmínky úvěru, ale **nejsou** promítnuty do výpočtu splátek — údaje pro ARM a buydown níže odrážejí jejich počáteční sazby 4,75 % a 4,50 % držené neměnné po celou dobu splácení. Považujte tyto dvě celkové částky za nejlepší možný (počáteční sazba) scénář, nikoli za nejhorší strop.

In [3]:
PROCEDURA loan START=2026:7 amount=27500 life=120 outsum=plan_sum;

   fixed rate=6.53
         label="Federální přímý nedotovaný úvěr (standardní)";

   fixed rate=5.99 init=412.50
         label="Soukromé refinancování (s poplatkem)";

   arm rate=4.75 caps=(1 5) maxrate=9.75 adjustfreq=12 worstcase
       label="Soukromý úvěr s proměnlivou sazbou (ARM, nejhorší případ)";

   buydown rate=4.50 bdrates=(13=5.50 25=6.50)
           label="Zaměstnavatelský buydown 2-1, poté 6,5 %";

   COMPARE at=(36 60 120) ALL taxrate=22 marr=4 OUTCOMP=plan_cmp;
SPUSTIT;

                            The LOAN Procedure

  Number of loans evaluated:    4

  Loan #1: Federální přímý nedotovaný úvěr (standardní)
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            6.5300
    Life (months):                          120
    Monthly Payment:                     312.68
    Total Paid (P&I):                  37521.22
    Total Interest:                    10021.22

  Loan #2: Soukromé refinancování (s poplatkem)
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            5.9900
    Life (months):                          120
    Monthly Payment:                     305.17
    Total Paid (P&I):                  36620.20
    Total Interest:                     9120.20
    Initialization Cost:                 412.50

  Loan #3: Soukromý úvěr s proměnlivou sazbou (ARM, nejhorší případ)
    Loan Type:                   Adjusta


NOTE: PROC LOAN loan analysis

NOTE: PROC LOAN statement used.


## Krok 4 — Samostatné spuštění doporučeného plánu s pevnou sazbou

Pro dlužníka, který oceňuje jistotu splátky, je federální standardní plán s pevnou sazbou bezpečnou výchozí volbou. Spustíme ho samostatně, abychom potvrdili jeho hlavní ukazatele: stejná splátka **312,68 USD** měsíčně, celkem zaplaceno **37 521,22 USD** a celkový úrok **10 021,22 USD**, jak vyplynulo ze čtyřstranného porovnání, nyní prezentované jako samostatný souhrn úvěru.

In [4]:
PROCEDURA loan START=2026:7 amount=27500 rate=6.53 life=120 schedule=yearly;
   fixed label="Federální přímý nedotovaný úvěr (standardní)";
SPUSTIT;

                            The LOAN Procedure

  Number of loans evaluated:    1

  Loan #1: Federální přímý nedotovaný úvěr (standardní)
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            6.5300
    Life (months):                          120
    Monthly Payment:                     312.68
    Total Paid (P&I):                  37521.22
    Total Interest:                    10021.22

  Loan Repayment Schedule: Federální přímý nedotovaný úvěr (standardní)
      Year    Begin Balance        Payment       Interest      Principal      End Balance
    ------ ---------------- -------------- -------------- -------------- ----------------
         1         27500.00        3752.12        1736.12        2016.00         25484.00
         2         25484.00        3752.12        1600.47        2151.66         23332.34
         3         23332.34        3752.12        1455.68        2296.44         21035.90
         4 


NOTE: PROC LOAN loan analysis

NOTE: PROC LOAN statement used.


## Interpretace výsledků

Všechny čtyři plány plně amortizují stejnou jistinu 27 500 USD během 120 měsíců (nesplacený zůstatek každého plánu dosáhne v tabulce COMPARE ve 120. měsíci 0,00 USD), takže rozhodnutí se odvíjí od **měsíční splátky** a **celkového úroku při uvedené sazbě**:

| Plán | Sazba | Splátka | Celkový úrok | Poznámky |
|------|-------|---------|----------------|-------|
| Federální standardní | 6,53 % | 312,68 USD | 10 021,22 USD | Bez poplatku; nejsilnější ochrana dlužníka |
| Soukromé refinancování | 5,99 % | 305,17 USD | 9 120,20 USD | Zřizovací poplatek 412,50 USD |
| Soukromý ARM | 4,75 % (start) | 288,33 USD | 7 099,76 USD | Sazba může růst; údaj je nákladem při počáteční sazbě |
| Zaměstnavatelský buydown 2-1 | 4,50 % (start) | 285,01 USD | 6 700,67 USD | Závisí na trvajícím zaměstnání |

- **Federální standardní** plán je z hlediska úroku nejdražší (10 021,22 USD), ale nabízí pevnou, předvídatelnou splátku 312,68 USD bez poplatku.
- **Soukromé refinancování** snižuje sazbu na 5,99 % (úrok 9 120,20 USD, o 901 USD méně než u federálního plánu), ale účtuje zřizovací poplatek 412,50 USD, takže jeho čistá výhoda oproti federálnímu plánu je zhruba 488 USD úroku minus poplatek — má smysl jen pokud úvěr běží dost dlouho na to, aby nebyl refinancován pryč.
- **ARM** a **buydown** vykazují zde nejnižší úrok (7 099,76 USD a 6 700,67 USD) čistě proto, že jejich **počáteční** sazby jsou výrazně nižší než u fixních nabídek. Jak bylo uvedeno v kroku 3, Jenner drží tyto počáteční sazby neměnné, takže jde o nejlepší možné údaje: reálný ARM, který se sazbou zvyšuje — nebo buydown, který ztratí dotaci zaměstnavatele — by dopadl hůře a splátka není zaručena.

Tabulka `COMPARE` toto zpřesňuje tím, že ukazuje, jak rychle se každý plán amortizuje. Ve 36. měsíci federální plán zaplatil 4 792,27 USD na úroku a splatil 6 464,10 USD jistiny (zůstatek 21 035,90 USD), zatímco buydown zaplatil jen 3 263,97 USD na úroku a splatil 6 996,24 USD jistiny (zůstatek 20 503,76 USD) — plány s nižší sazbou stojí méně na úroku a zároveň rychleji ukrajují z jistiny během prvních tří let. Pro rizikově averzního absolventa často jistota sazby federálního standardního plánu ospravedlní jeho vyšší úrok; pro dlužníka, který si je jistý stabilním, trvalým zaměstnáním, přináší dotovaný start buydownu největší úsporu mezi možnostmi, které si skutečně fixují sazbu.